# Chapter 3 실습: 어텐션 메커니즘 구현하기

교재 Chapter 3 흐름을 따라 **Dot Product Attention → Self-Attention → Q/K/V Attention → Causal Attention → Multi-Head Attention**을 직접 구현합니다.

## 실습 목표
1. Attention score, attention weight, context vector의 차이를 이해한다.
2. 행렬곱으로 Self-Attention을 구현한다.
3. Q, K, V를 사용하는 학습 가능한 Self-Attention을 구현한다.
4. GPT 계열 모델에서 필요한 causal mask를 구현한다.
5. Multi-Head Attention의 출력 shape를 이해한다.

## 진행 방식
- 배포용 노트북은 `TODO`를 채우는 방식입니다.
- 운영진용 노트북에는 정답 코드와 검증 코드가 들어 있습니다.
- 각 파트 마지막의 검증 셀을 통과해야 다음 단계로 넘어가는 것을 추천합니다.

In [5]:
import torch
import torch.nn as nn

# 재현성을 위한 seed 고정
torch.manual_seed(123)

# 교재 예제 문장 기반 입력 임베딩
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55], # step
])

tokens = ["Your", "journey", "starts", "with", "one", "step"]
print("inputs.shape:", inputs.shape)

inputs.shape: torch.Size([6, 3])


## Part 1. Dot Product Attention 직접 구현하기

`journey` 토큰을 query로 두고 모든 토큰과 dot product를 계산해 attention score를 만듭니다.

In [4]:
def get_attention_scores(inputs, query_index):
    """
    inputs: [num_tokens, embedding_dim]
    query_index: 기준이 되는 토큰 위치
    return: attn_scores [num_tokens]
    """
    # TODO 1. query_index에 해당하는 query 벡터를 가져오세요.
    query = inputs[query_index]

    # TODO 2. attention score를 저장할 빈 텐서를 만드세요.
    attn_scores = torch.zeros(inputs.shape[0])

    # TODO 3. 모든 입력 벡터와 query의 dot product를 계산하세요.
    for i, x_i in enumerate(inputs):
        attn_scores[i] = torch.dot(query, x_i)

    return attn_scores

scores = get_attention_scores(inputs, query_index=1)
print(scores)


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [3]:
expected_scores = torch.tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
assert scores.shape == torch.Size([6])
assert torch.allclose(scores, expected_scores, atol=1e-4)
print("Part 1 통과")

Part 1 통과


## Part 2. Attention Weight와 Context Vector 구하기

Attention score에 softmax를 적용해 attention weight를 만들고, 입력 벡터들의 가중합으로 context vector를 계산합니다.

In [6]:
def get_context_vector(inputs, query_index):
    # TODO 1. attention score 계산
    attn_scores = get_attention_scores(inputs, query_index)

    # TODO 2. softmax로 attention weight 계산
    attn_weights = torch.softmax(attn_scores, dim=0)

    # TODO 3. weighted sum으로 context vector 계산
    context_vec = torch.sum(attn_weights.unsqueeze(-1) * inputs, dim=0)

    return context_vec, attn_weights

context_vec, attn_weights = get_context_vector(inputs, query_index=1)
print("attention weights:", attn_weights)
print("sum:", attn_weights.sum())
print("context vector:", context_vec)


attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
sum: tensor(1.)
context vector: tensor([0.4419, 0.6515, 0.5683])


In [7]:
assert attn_weights.shape == torch.Size([6])
assert torch.allclose(attn_weights.sum(), torch.tensor(1.0), atol=1e-6)
assert context_vec.shape == torch.Size([3])
expected_context = torch.tensor([0.4419, 0.6515, 0.5683])
assert torch.allclose(context_vec, expected_context, atol=1e-4)
print("Part 2 통과")

Part 2 통과


## Part 3. 모든 토큰에 대해 Self-Attention 계산하기

`inputs @ inputs.T`로 모든 토큰 쌍의 attention score matrix를 한 번에 계산합니다.

In [8]:
def self_attention_basic(inputs):
    """
    trainable weight가 없는 기본 self-attention 구현
    return: attn_weights [num_tokens, num_tokens], context_vecs [num_tokens, embedding_dim]
    """
    # TODO 1. attention score matrix 계산
    attn_scores = inputs @ inputs.T

    # TODO 2. 각 row마다 softmax 적용
    attn_weights = torch.softmax(attn_scores, dim=-1)

    # TODO 3. context vector matrix 계산
    context_vecs = attn_weights @ inputs

    return attn_weights, context_vecs

attn_weights_basic, context_vecs_basic = self_attention_basic(inputs)
print("attn_weights_basic.shape:", attn_weights_basic.shape)
print("context_vecs_basic.shape:", context_vecs_basic.shape)
print("row sums:", attn_weights_basic.sum(dim=-1))


attn_weights_basic.shape: torch.Size([6, 6])
context_vecs_basic.shape: torch.Size([6, 3])
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [9]:
assert attn_weights_basic.shape == torch.Size([6, 6])
assert context_vecs_basic.shape == torch.Size([6, 3])
assert torch.allclose(attn_weights_basic.sum(dim=-1), torch.ones(6), atol=1e-6)
assert torch.allclose(context_vecs_basic[1], torch.tensor([0.4419, 0.6515, 0.5683]), atol=1e-4)
print("Part 3 통과")

Part 3 통과


## Part 4. Q, K, V를 사용하는 Self-Attention 구현하기

입력 벡터를 학습 가능한 가중치 행렬로 Query, Key, Value로 변환합니다. 이후 scaled dot-product attention을 계산합니다.

In [10]:
def self_attention_qkv_manual(inputs, d_out=2):
    torch.manual_seed(123)
    d_in = inputs.shape[1]

    W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
    W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
    W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

    # TODO 1. queries, keys, values 계산
    queries = inputs @ W_query
    keys = inputs @ W_key
    values = inputs @ W_value

    # TODO 2. attention scores 계산
    attn_scores = queries @ keys.T

    # TODO 3. scaled dot-product attention 적용
    d_k = keys.shape[-1]
    attn_weights = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)

    # TODO 4. context vectors 계산
    context_vecs = attn_weights @ values

    return context_vecs, attn_weights, queries, keys, values

context_vecs_qkv, attn_weights_qkv, queries, keys, values = self_attention_qkv_manual(inputs)
print("queries.shape:", queries.shape)
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)
print("attn_weights_qkv.shape:", attn_weights_qkv.shape)
print("context_vecs_qkv.shape:", context_vecs_qkv.shape)


queries.shape: torch.Size([6, 2])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
attn_weights_qkv.shape: torch.Size([6, 6])
context_vecs_qkv.shape: torch.Size([6, 2])


In [11]:
assert queries.shape == torch.Size([6, 2])
assert keys.shape == torch.Size([6, 2])
assert values.shape == torch.Size([6, 2])
assert attn_weights_qkv.shape == torch.Size([6, 6])
assert context_vecs_qkv.shape == torch.Size([6, 2])
assert torch.allclose(attn_weights_qkv.sum(dim=-1), torch.ones(6), atol=1e-6)
print("Part 4 통과")

Part 4 통과


## Part 5. `nn.Module`로 SelfAttention 클래스 만들기

`nn.Linear`를 사용해 재사용 가능한 Self-Attention 모듈을 구현합니다.

In [13]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        # keys, queries, values 계산
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # attention scores 계산 (2D inputs: [num_tokens, d])
        attn_scores = queries @ keys.T

        # attention weights 계산
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # context vectors 계산
        context_vecs = attn_weights @ values

        return context_vecs

torch.manual_seed(789)
sa = SelfAttention(d_in=3, d_out=2)
out_sa = sa(inputs)
print(out_sa)
print(out_sa.shape)


tensor([[-0.0726,  0.0731],
        [-0.0740,  0.0716],
        [-0.0740,  0.0715],
        [-0.0756,  0.0692],
        [-0.0760,  0.0685],
        [-0.0748,  0.0703]], grad_fn=<MmBackward0>)
torch.Size([6, 2])


In [14]:
assert out_sa.shape == torch.Size([6, 2])
print("Part 5 통과")

Part 5 통과


## Part 6. Causal Mask 적용하기

GPT 계열 모델은 현재 토큰보다 미래에 있는 토큰을 보면 안 됩니다. 미래 위치를 `-inf`로 바꾼 뒤 softmax를 적용합니다.

In [15]:
def apply_causal_mask(attn_scores):
    context_length = attn_scores.shape[0]

    # TODO 1. upper triangular mask 생성
    mask = torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()

    # TODO 2. mask 위치를 -inf로 채우기
    masked_scores = attn_scores.masked_fill(mask, float('-inf'))

    return masked_scores

attn_scores_qkv = queries @ keys.T
masked_scores = apply_causal_mask(attn_scores_qkv)
masked_weights = torch.softmax(masked_scores / keys.shape[-1]**0.5, dim=-1)
print(masked_weights)
print("row sums:", masked_weights.sum(dim=-1))


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3986, 0.6014, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2526, 0.3791, 0.3683, 0.0000, 0.0000, 0.0000],
        [0.2265, 0.2839, 0.2794, 0.2103, 0.0000, 0.0000],
        [0.1952, 0.2363, 0.2331, 0.1820, 0.1534, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [16]:
assert masked_weights.shape == torch.Size([6, 6])
assert torch.allclose(masked_weights.sum(dim=-1), torch.ones(6), atol=1e-6)
assert torch.all(masked_weights[0, 1:] == 0)
assert torch.all(masked_weights[1, 2:] == 0)
assert torch.all(masked_weights[2, 3:] == 0)
print("Part 6 통과")

Part 6 통과


## Part 7. Attention Dropout 적용하기

학습 중 일부 attention 연결을 무작위로 제거해 특정 연결에 과도하게 의존하는 것을 줄입니다.

In [17]:
def apply_attention_dropout(attn_weights, dropout_rate=0.5):
    torch.manual_seed(123)

    # TODO 1. Dropout 객체 생성
    dropout = nn.Dropout(dropout_rate)

    # TODO 2. attention weight에 dropout 적용
    dropped_weights = dropout(attn_weights)

    return dropped_weights

for p in [0.0, 0.1, 0.5]:
    dropped = apply_attention_dropout(masked_weights, dropout_rate=p)
    print(f"dropout={p}")
    print(dropped)


dropout=0.0
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3986, 0.6014, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2526, 0.3791, 0.3683, 0.0000, 0.0000, 0.0000],
        [0.2265, 0.2839, 0.2794, 0.2103, 0.0000, 0.0000],
        [0.1952, 0.2363, 0.2331, 0.1820, 0.1534, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])
dropout=0.1
tensor([[1.1111, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4428, 0.6683, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2807, 0.4212, 0.4092, 0.0000, 0.0000, 0.0000],
        [0.2516, 0.3154, 0.3104, 0.2337, 0.0000, 0.0000],
        [0.2169, 0.2626, 0.0000, 0.2022, 0.1704, 0.0000],
        [0.1730, 0.2324, 0.2276, 0.1577, 0.1210, 0.1994]])
dropout=0.5
tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5052, 0.7582, 0.7366, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5677, 0.5587, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4727, 0.0000, 0

In [18]:
assert apply_attention_dropout(masked_weights, 0.0).shape == masked_weights.shape
print("Part 7 통과")

Part 7 통과


## Part 8. CausalAttention 클래스 완성하기

Q/K/V, causal mask, dropout을 모두 포함한 GPT 스타일 attention 모듈입니다. 입력 shape는 `[batch_size, num_tokens, d_in]`입니다.

In [19]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # TODO 1. keys, queries, values 계산
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # TODO 2. attention scores 계산. 힌트: keys.transpose(1, 2)
        attn_scores = queries @ keys.transpose(1, 2)

        # TODO 3. causal mask 적용
        mask = self.mask[:num_tokens, :num_tokens].bool()
        attn_scores = attn_scores.masked_fill(mask.unsqueeze(0), float('-inf'))

        # TODO 4. softmax 적용
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)

        # TODO 5. dropout 적용
        attn_weights = self.dropout(attn_weights)

        # TODO 6. context vectors 계산
        context_vecs = attn_weights @ values
        return context_vecs

batch = torch.stack((inputs, inputs), dim=0)
torch.manual_seed(123)
ca = CausalAttention(d_in=3, d_out=2, context_length=batch.shape[1], dropout=0.0)
context_vecs_ca = ca(batch)
print(context_vecs_ca)
print(context_vecs_ca.shape)


tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 6, 2])


In [20]:
assert context_vecs_ca.shape == torch.Size([2, 6, 2])
print("Part 8 통과")

Part 8 통과


## Part 9. Multi-Head Attention 구현하기: Wrapper 방식

여러 개의 CausalAttention head를 병렬로 실행한 뒤 마지막 차원 기준으로 concat합니다.

In [21]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            for _ in range(num_heads)
        ])

    def forward(self, x):
        # 각 head의 출력을 마지막 차원 기준으로 concat
        outputs = [h(x) for h in self.heads]
        return torch.cat(outputs, dim=-1)

torch.manual_seed(123)
mha_wrapper = MultiHeadAttentionWrapper(d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2)
context_vecs_mha_wrapper = mha_wrapper(batch)
print(context_vecs_mha_wrapper)
print(context_vecs_mha_wrapper.shape)


tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
torch.Size([2, 6, 4])


In [22]:
assert context_vecs_mha_wrapper.shape == torch.Size([2, 6, 4])
print("Part 9 통과")

Part 9 통과


## Part 10. 효율적인 MultiHeadAttention 구현하기

Q/K/V를 한 번에 계산한 뒤 `view`, `transpose`로 head를 나눕니다. 실제 Transformer 구현에 더 가까운 방식입니다.

In [23]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어 떨어져야 합니다."
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # TODO 1. Q/K/V 계산: [b, num_tokens, d_out]
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # TODO 2. head 분리: [b, num_tokens, num_heads, head_dim]
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # TODO 3. transpose: [b, num_heads, num_tokens, head_dim]
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # TODO 4. attention scores 계산: [b, num_heads, num_tokens, num_tokens]
        attn_scores = queries @ keys.transpose(-2, -1)

        # TODO 5. causal mask 적용
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask_bool.unsqueeze(0).unsqueeze(0), float('-inf'))

        # TODO 6. softmax + dropout
        attn_weights = torch.softmax(attn_scores / (self.head_dim ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # TODO 7. context 계산 후 shape 복원
        context_vec = attn_weights @ values
        context_vec = context_vec.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)

        # TODO 8. output projection
        context_vec = self.out_proj(context_vec)
        return context_vec

torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2)
context_vecs_mha = mha(batch)
print(context_vecs_mha)
print(context_vecs_mha.shape)


tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
torch.Size([2, 6, 4])


In [24]:
assert context_vecs_mha.shape == torch.Size([2, 6, 4])
print("Part 10 통과")

Part 10 통과


# 최종 과제: Tiny GPT Attention Block

아래 조건을 만족하는 GPT 스타일 attention block을 실행하세요.

## 조건
- 입력 shape: `[batch_size, context_length, d_in]`
- 출력 shape: `[batch_size, context_length, d_out]`
- causal mask 적용
- multi-head attention 사용

## 제출 체크리스트
1. 모든 검증 셀 통과
2. `dropout=0.0`, `dropout=0.1` 결과 비교
3. `num_heads=1`, `num_heads=2`, `num_heads=4` shape 비교
4. README에 Q/K/V, causal mask, multi-head attention을 자신의 말로 설명

In [25]:
x = torch.randn(4, 8, 16)

model = MultiHeadAttention(
    d_in=16,
    d_out=16,
    context_length=8,
    dropout=0.1,
    num_heads=4
)

out = model(x)
print(out.shape)
assert out.shape == torch.Size([4, 8, 16])
print("최종 과제 기본 실행 통과")

torch.Size([4, 8, 16])
최종 과제 기본 실행 통과


2. `dropout=0.0`, `dropout=0.1` 결과 비교
드롭아웃은 신경망 훈련중 은닉층의 유닛을 랜덤하게 선택하여 무시함으로써 과도하게 의존하는 과대 적합을 막는 기법입니다
dropout=0일때는 드롭아웃이 전혀 적용되지않고 
0.1일때는 어텐션 가중치가 약 10%로 랜덤하게 0으로 바뀝니다 이떄 중요한점은 삭제된 값들을 보상하기위해 살아남은 나머지 가중치들의 값이 그 비율만큼 증가한다는 것입니다

3. `num_heads=1`, `num_heads=2`, `num_heads=4` shape 비교
멀티 해드 어텐셔능ㄴ 어텐션 메커니즘을 여러개의 독립적인 헤드로 나누어 병렬로 연산합니다 효율적인 연산을 위해서 큰 행렬 하나를 만든뒤 이를 헤드 개수 만큼 논리적으로 쪼개서 연산합니다num_head가 1,2,4로 늘어나더라도 각 헤드가 담당하는 차원은 그에 비례해서 작아집니다 연산이 끝난후 다시 모든 헤드를 하나로 이어 붙여 최종적으로는 처음과 같은 크기가 됩니다

4. README에 Q/K/V, causal mask, multi-head attention을 자신의 말로 설명
Q는 쿼리로 초점을 맞추고자 하는ㄴ 기준 단어 Key는 문장내 다른단어들이 자신이 어떤 정보를 가지고 있는지 나타내는 라벨 value는 각 단어가 실제로 품고있는 진짜 의미나 정보 입니다 코잘 마스크는 미래 가림막으로 다음 단어 예측하는 훈련을 할때 미래 나올 단어를 못보게 막는 역할을 합니다 멀티헤드 어텐션은 한번에 문장을 훑어보는 대신에 여러개의 관점을 만들어 문장을 동시에 다각도로 분석하는 기술입니다